# 02. Initial data description: `ami_meter_clean`


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

from bms_sa_review.synthetic_ami_creation.config import ami_config as Config
from bms_sa_review.synthetic_ami_creation.lib import ami_plots as Plots

sns.set_theme(style="ticks", context="notebook")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Validated palette (see the `dataviz` skill): fixed categorical hue order
# (8 hues, CVD-checked adjacent ordering), a dedicated neutral for "Other",
# and a single blue sequential ramp for every magnitude encoding below --
# never a rainbow, never color re-used to mean two different things.
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
OTHER_COLOR = "#9a9a94"
SEQUENTIAL_HEX = ["#cde2fb", "#86b6ef", "#3987e5", "#2a78d6", "#1c5cab", "#0d366b"]
SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQUENTIAL_HEX)
ACCENT = CATEGORICAL[0]

con = duckdb.connect()
con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")
con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_clean AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_meter_clean").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")
con.sql(f"""
    CREATE OR REPLACE VIEW ami_site_metadata AS
    SELECT * FROM read_parquet('{Config.store_path("ami_site_metadata").as_posix()}')
""")

# THE site scope for this whole notebook: only sites actually present in
# ami_meter_clean, not ami_site_metadata's own (broader) in_ami_meter flag.
con.sql("""
    CREATE OR REPLACE VIEW ami_site_metadata_clean AS
    SELECT m.*
    FROM ami_site_metadata m
    INNER JOIN (SELECT DISTINCT site_id FROM ami_meter_clean) c USING (site_id)
""")


## 1. Fleet overview


In [ ]:
n_sites_metadata = con.sql("SELECT count(*) FROM ami_site_metadata").fetchone()[0]
n_sites_clean = con.sql("SELECT count(*) FROM ami_site_metadata_clean").fetchone()[0]
n_rows_clean = con.sql("SELECT count(*) FROM ami_meter_clean").fetchone()[0]
n_rows_raw_meter = con.sql("SELECT count(*) FROM ami_meter").fetchone()[0]

print(f"{n_sites_metadata:,} sites total in ami_site_metadata.")
print(f"{n_sites_clean:,} of those have at least one row in ami_meter_clean "
      f"-- this is the site scope for every section below.")
print(f"\nami_meter:       {n_rows_raw_meter:,} rows")
print(f"ami_meter_clean: {n_rows_clean:,} rows "
      f"({1 - n_rows_clean / n_rows_raw_meter:.4%} removed by notebook 01's hard checks)")


## 1.2. Metadata overview

In [ ]:
con.sql("""
-- See the table's columns
DESCRIBE ami_site_metadata_clean;

-- See a few sample rows
SELECT *
FROM ami_site_metadata_clean
LIMIT 10;
""")

In [ ]:
con.sql("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'ami_site_metadata_clean'
ORDER BY ordinal_position;
""")

## 2. Geographic distribution

By DNSP and by site location, restricted to `ami_site_metadata_clean`. Every
DNSP with more than `DNSP_MIN_SITES` sites gets its own color (same mapping
in both charts, so identity stays consistent between them); the rest fold
into "Other". The full per-DNSP table (all of them, not just the ones drawn
individually) is printed alongside the chart.


In [ ]:
DNSP_MIN_SITES = 100

dnsp_counts = con.sql("""
    SELECT COALESCE(dnsp_name, 'Unknown') AS dnsp_name, count(*) AS n_sites
    FROM ami_site_metadata_clean GROUP BY COALESCE(dnsp_name, 'Unknown') ORDER BY n_sites DESC
""").df()

# dnsp_counts is already sorted by n_sites desc, so this preserves rank order.
candidates = dnsp_counts.loc[dnsp_counts.n_sites > DNSP_MIN_SITES, "dnsp_name"].tolist()
if len(candidates) > len(CATEGORICAL):
    print(f"{len(candidates)} DNSPs exceed {DNSP_MIN_SITES} sites -- more than the "
          f"{len(CATEGORICAL)} reliably distinguishable colors available. Showing only "
          f"the top {len(CATEGORICAL)} by site count individually; the rest (each still "
          f">{DNSP_MIN_SITES} sites) fold into 'Other' too -- see the full table below "
          "for their real counts.")
big_dnsp = candidates[:len(CATEGORICAL)]

dnsp_counts["dnsp_group"] = dnsp_counts.dnsp_name.where(dnsp_counts.dnsp_name.isin(big_dnsp), "Other")
dnsp_order = big_dnsp + (["Other"] if (~dnsp_counts.dnsp_name.isin(big_dnsp)).any() else [])

dnsp_palette = dict(zip(big_dnsp, CATEGORICAL[:len(big_dnsp)]))
if "Other" in dnsp_order:
    dnsp_palette["Other"] = OTHER_COLOR

grouped_counts = dnsp_counts.groupby("dnsp_group", as_index=False).n_sites.sum()
grouped_counts["dnsp_group"] = pd.Categorical(grouped_counts.dnsp_group, categories=dnsp_order, ordered=True)
grouped_counts = grouped_counts.sort_values("dnsp_group")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(grouped_counts.dnsp_group.astype(str), grouped_counts.n_sites,
       color=[dnsp_palette[d] for d in grouped_counts.dnsp_group])
ax.set_ylabel("Sites")
ax.set_title(f"Sites by DNSP (top {len(big_dnsp)} shown individually, each > {DNSP_MIN_SITES} sites)")
sns.despine(ax=ax)
fig.tight_layout()

print(f"Full breakdown, all {len(dnsp_counts)} DNSPs (ami_meter_clean scope):")
dnsp_counts[["dnsp_name", "n_sites"]].style.format({"n_sites": "{:,}"})

In [ ]:
site_locations = con.sql("""
    SELECT site_id, COALESCE(dnsp_name, 'Unknown') AS dnsp_name, longitude, latitude
    FROM ami_site_metadata_clean
    WHERE longitude IS NOT NULL AND latitude IS NOT NULL
""").df()
site_locations["dnsp_group"] = site_locations.dnsp_name.where(site_locations.dnsp_name.isin(big_dnsp), "Other")

fig, ax = plt.subplots(figsize=(6, 6))
for group in dnsp_order:
    subset = site_locations[site_locations.dnsp_group == group]
    ax.scatter(subset.longitude, subset.latitude, s=8, alpha=0.5,
               color=dnsp_palette[group], label=f"{group} ({len(subset):,})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Site locations by DNSP (ami_meter_clean scope)")
ax.set_aspect("equal")
ax.legend(frameon=False, markerscale=2, loc="upper left", bbox_to_anchor=(1.02, 1))
sns.despine(ax=ax)
fig.tight_layout()

print(f"{len(site_locations):,} of {n_sites_clean:,} sites (in ami_meter_clean) have a non-null location.")


## 3. Data coverage (P, Q, V)

**Reading the heatmap:** rows are sorted by TOTAL months present, which is
not the same as "starts full at the top, decays monotonically toward the
bottom". A site can have a below-average total yet still show a solid block
on the RIGHT (2025) with nothing on the left (2024) -- that's a site whose
PV/meter was only installed partway through the study period, not one that
dropped out early. Coverage loss in this fleet comes from both directions
(late starts as well as early endings, plus some mid-series gaps), so the
picture is a genuine mosaic, not a clean wedge -- sorting by total captures
"how much" coverage a site has, not "when" it has it.


In [ ]:
null_shares = con.sql("""
    SELECT
      1 - avg(CASE WHEN V IS NULL THEN 1.0 ELSE 0.0 END) AS V_coverage,
      1 - avg(CASE WHEN P_kw IS NULL THEN 1.0 ELSE 0.0 END) AS P_kw_coverage,
      1 - avg(CASE WHEN Q_kvar IS NULL THEN 1.0 ELSE 0.0 END) AS Q_kvar_coverage
    FROM ami_meter_clean
""").df()
(null_shares.T.rename(columns={0: "share_non_null"}) * 1).style.format("{:.4%}")

In [ ]:
coverage_by_month = con.sql("""
    SELECT site_id, year, month, count(*) AS n_rows
    FROM ami_meter_clean GROUP BY site_id, year, month
""").df()
coverage_by_month["ym"] = (
    coverage_by_month.year.astype(str) + "-" + coverage_by_month.month.astype(str).str.zfill(2)
)
pivot = coverage_by_month.pivot_table(index="site_id", columns="ym", values="n_rows", fill_value=0)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]  # sorted by total coverage

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap((pivot.values > 0), cmap=SEQUENTIAL_CMAP, cbar=False, ax=ax)
ax.set_xticks([i + 0.5 for i in range(len(pivot.columns))])
ax.set_xticklabels(pivot.columns, rotation=90, fontsize=7)
ax.set_yticks([])
ax.set_xlabel("Month")
ax.set_ylabel(f"Site (n={len(pivot):,}, sorted by total coverage)")
ax.set_title("Data coverage by site and month (ami_meter_clean)")
fig.tight_layout()


## 4. Range and distribution of P, Q, V


In [ ]:
full_range = con.sql("""
    SELECT
      min(V) AS V_min, max(V) AS V_max,
      min(P_kw) AS P_kw_min, max(P_kw) AS P_kw_max,
      min(Q_kvar) AS Q_kvar_min, max(Q_kvar) AS Q_kvar_max
    FROM ami_meter_clean
""").df()

sample = con.sql("SELECT t_stamp, year, month, V, P_kw, Q_kvar FROM ami_meter_clean USING SAMPLE 2000000 ROWS").df()

summary_table = pd.DataFrame({
    "min": [full_range.V_min[0], full_range.P_kw_min[0], full_range.Q_kvar_min[0]],
    "max": [full_range.V_max[0], full_range.P_kw_max[0], full_range.Q_kvar_max[0]],
    "mean (sample)": [sample.V.mean(), sample.P_kw.mean(), sample.Q_kvar.mean()],
    "median (sample)": [sample.V.median(), sample.P_kw.median(), sample.Q_kvar.median()],
}, index=["V", "P_kw", "Q_kvar"])
summary_table.style.format("{:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, col, label in zip(axes, ["V", "P_kw", "Q_kvar"], ["Voltage (V)", "Active power (kW)", "Reactive power (kVAr)"]):
    ax.hist(sample[col].dropna(), bins=80, color=ACCENT)
    ax.set_xlabel(label)
    ax.set_ylabel("Count" if ax is axes[0] else "")
    if col in ("P_kw", "Q_kvar"):
        ax.set_yscale("log")
    sns.despine(ax=ax)
fig.suptitle("Distributions (2M-row sample)")
fig.tight_layout()


## 5. Voltage distribution vs AS/NZS 4777.2:2020 reference points

`V` restricted to 200--300V, with the standard's "Australia A" DEFAULT
Volt-Watt and Volt-VAr reference voltages marked. **These are the
standard's default set-points, not necessarily what your specific DNSP(s)
actually approved/enabled** -- confirm against the real regional settings
before using this for a compliance claim in the paper; treat it as an
orientational view of where the data sits relative to the standard, not a
compliance verdict.


In [ ]:
# AS/NZS 4777.2:2020 Australia A default set-points
AS4777 = {
    # Volt-Watt: 100% S_rated at/below V1, ramp to 20% at V2
    "VW": {"V1": 253.0, "V2": 260.0},
    # Volt-VAr: +Q1 supplying <=V1, 0 across deadband V2..V3, -Q4 absorbing >=V4
    "VVAR": {"V1": 207.0, "V2": 220.0, "V3": 240.0, "V4": 258.0},
}
vw_color, vvar_color = CATEGORICAL[1], CATEGORICAL[2]
threshold_lines = [
    ("VVAr V1", AS4777["VVAR"]["V1"], vvar_color, 1.00),
    ("VVAr V2", AS4777["VVAR"]["V2"], vvar_color, 1.00),
    ("VVAr V3", AS4777["VVAR"]["V3"], vvar_color, 1.00),
    ("VW V1",   AS4777["VW"]["V1"],   vw_color,   1.00),
    ("VVAr V4", AS4777["VVAR"]["V4"], vvar_color, 1.12),
    ("VW V2",   AS4777["VW"]["V2"],   vw_color,   1.24),
]

v_sample = sample.V.dropna()
v_sample = v_sample[(v_sample >= 200) & (v_sample <= 300)]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.hist(v_sample, bins=100, range=(200, 300), color=ACCENT)
ymax = ax.get_ylim()[1]
for label, v, color, height_mult in threshold_lines:
    ax.axvline(v, color=color, linestyle="--", linewidth=1.2)
    ax.text(v, ymax * height_mult, label, color=color, rotation=90,
            ha="center", va="bottom", fontsize=8, clip_on=False)
ax.set_ylim(top=ymax * 1.4)
ax.set_xlim(200, 300)
ax.set_xlabel("Voltage (V)")
ax.set_ylabel("Count (2M-row sample)")
ax.set_title("Voltage distribution vs AS/NZS 4777.2:2020 default reference points")
sns.despine(ax=ax)
fig.tight_layout()

exposure = con.sql(f"""
    SELECT
      avg(CASE WHEN V <= {AS4777["VVAR"]["V1"]} THEN 1.0 ELSE 0.0 END) AS share_at_or_below_vvar_v1,
      avg(CASE WHEN V >= {AS4777["VW"]["V1"]} THEN 1.0 ELSE 0.0 END) AS share_at_or_above_vw_v1,
      avg(CASE WHEN V >= {AS4777["VVAR"]["V4"]} THEN 1.0 ELSE 0.0 END) AS share_at_or_above_vvar_v4
    FROM ami_meter_clean WHERE V IS NOT NULL
""").df()
exposure.style.format("{:.4%}")


## 6. Diurnal load profile (load-only -- see the note at the top)

Average `P_kw` by hour of day, in local time (fixed AEST, no DST -- same
convention as `ami_plots.to_aest`), weekday vs weekend.


In [ ]:
sample_local = sample.copy()
sample_local["t_stamp_local"] = Plots.to_aest(sample_local.t_stamp)
sample_local["hour"] = sample_local.t_stamp_local.dt.hour
sample_local["is_weekend"] = sample_local.t_stamp_local.dt.dayofweek >= 5

diurnal = (
    sample_local.groupby(["hour", "is_weekend"])["P_kw"].mean().reset_index()
)

fig, ax = plt.subplots(figsize=(7, 3.5))
for is_weekend, label, color in [(False, "Weekday", CATEGORICAL[0]), (True, "Weekend", CATEGORICAL[1])]:
    subset = diurnal[diurnal.is_weekend == is_weekend].sort_values("hour")
    ax.plot(subset.hour, subset.P_kw, marker="o", markersize=3, color=color, label=label)
ax.set_xlabel("Hour of day (local)")
ax.set_ylabel("Mean P_kw")
ax.set_title("Average diurnal load profile (2M-row sample)")
ax.set_xticks(range(0, 24, 3))
ax.legend(frameon=False)
sns.despine(ax=ax)
fig.tight_layout()


## 7. Seasonal variation in load (load-only)

Average `P_kw` by calendar month, using the table's own `month` column
(assigned at build time from the source UTC timestamp -- a day or two of
edge-of-month drift near midnight UTC is an existing, accepted
simplification in this pipeline, not new here).


In [ ]:
seasonal = con.sql("""
    SELECT year, month, avg(P_kw) AS mean_P_kw
    FROM ami_meter_clean WHERE P_kw IS NOT NULL
    GROUP BY year, month ORDER BY year, month
""").df()
seasonal["ym"] = seasonal.year.astype(str) + "-" + seasonal.month.astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(seasonal.ym, seasonal.mean_P_kw, marker="o", markersize=4, color=ACCENT)
ax.set_xlabel("Month")
ax.set_ylabel("Mean P_kw")
ax.set_title("Seasonal variation in mean load")
ax.tick_params(axis="x", rotation=90)
sns.despine(ax=ax)
fig.tight_layout()


## 8. Export limit

`export_limit_kw`, site scope = `ami_site_metadata_clean`. Just the count
here -- how binding these limits are against actual PV output is a question
for the ground-truth (`ami_raw`) tables, out of scope for this notebook.


In [ ]:
n_export_limit = con.sql("""
    SELECT count(*) FROM ami_site_metadata_clean WHERE export_limit_kw IS NOT NULL
""").fetchone()[0]
print(f"{n_export_limit:,} of {n_sites_clean:,} sites have a non-null export_limit_kw.")


## 9. PV install date, fleet vintage, and system size

Fleet vintage matters for a compliance study specifically because older
inverters may predate AS/NZS 4777.2:2020 (or 4777.2 entirely) and may not
implement Volt-Watt/Volt-VAr response at all -- a real covariate, not just
background color. System size below uses `ac_capacity_kw` (nameplate AC
rating) as the "size" -- confirm that's the definition you want (the
alternative, `dc_capacity_kw`, is also in `ami_site_metadata_clean`).


In [ ]:
install_dates = con.sql("""
    SELECT pv_install_date, ac_capacity_kw
    FROM ami_site_metadata_clean WHERE pv_install_date IS NOT NULL
""").df()
install_dates["install_year"] = pd.to_datetime(install_dates.pv_install_date).dt.year

# System size buckets: 1kW-wide from 0 to 30kW, then a single 30kW+ aggregate.
BUCKET_EDGES = list(range(0, 31, 1))
bucket_labels = [f"{lo}-{lo + 1}" for lo in BUCKET_EDGES[:-1]] + ["30+"]
install_dates["size_bucket"] = pd.cut(
    install_dates.ac_capacity_kw, bins=BUCKET_EDGES + [float("inf")],
    labels=bucket_labels, right=False, include_lowest=True,
)

# Both panels must show the exact same set of years, in the same order, on
# the same x positions -- computed once here rather than left to each
# panel's own groupby, since a year could otherwise appear in one panel but
# not the other (e.g. every install in a year happens to have a null
# ac_capacity_kw, which pd.cut turns into NaN and groupby then drops).
ALL_YEARS = sorted(install_dates.install_year.unique())

by_year = (
    install_dates.groupby("install_year").size()
    .reindex(ALL_YEARS, fill_value=0)
)

# Rows = size bucket, columns = install year, cells = site count -- built as a
# full cross-tab (not just the combinations that happen to occur) so an
# empty year/bucket combination shows as zero, not a missing column/row.
size_by_year = (
    install_dates.groupby(["size_bucket", "install_year"], observed=False)
    .size()
    .reset_index(name="n_sites")
    .pivot(index="size_bucket", columns="install_year", values="n_sites")
    .reindex(index=bucket_labels, columns=ALL_YEARS, fill_value=0)
    .astype(int)
)

# height_ratios controls the relative height of the two panels -- raised
# from [1, 2] to [1, 4] to shrink the bar chart and grow the heatmap.
fig, (ax_year, ax_size) = plt.subplots(
    2, 1, figsize=(8, 10), gridspec_kw={"height_ratios": [1, 4]}, sharex=True,
)

# Bar chart plotted on the same integer x positions the heatmap columns will
# occupy (seaborn centres heatmap column i at i + 0.5), rather than on the
# year labels themselves -- this is what actually makes `sharex` line the
# two panels' columns up, not just their axis ranges.
x_pos = [i + 0.5 for i in range(len(ALL_YEARS))]
ax_year.bar(x_pos, by_year.values, width=0.8, color=ACCENT)
ax_year.set_ylabel("Sites")
ax_year.set_title("PV fleet vintage and system size by install year")
sns.despine(ax=ax_year, bottom=True)
ax_year.tick_params(axis="x", bottom=False, labelbottom=False)

# `yticklabels=1` forces every size bucket to get a label -- seaborn's
# default ("auto") thins them out once there are this many rows.
# `pad`/`fraction` trimmed down from their earlier generous values: `pad`
# is the gap between the heatmap and the colorbar, `fraction` is the total
# vertical band reserved for pad + colorbar together, and both were larger
# than a slim colorbar actually needs, which is what left the whitespace.
sns.heatmap(
    size_by_year, cmap=SEQUENTIAL_CMAP, ax=ax_size, yticklabels=1,
    cbar_kws={
        "label": "Sites", "orientation": "horizontal", "location": "bottom",
        "pad": 0.12, "fraction": 0.05, "shrink": 0.5, "aspect": 40,
    },
)
ax_size.set_xlabel("PV install year")
ax_size.set_ylabel("Nameplate system size, ac_capacity_kw (kW)")
ax_size.set_xticklabels([str(y) for y in ALL_YEARS], rotation=90)
ax_size.invert_yaxis()  # smallest bucket at the bottom, largest at the top

fig.tight_layout()

print(f"{len(install_dates):,} of {n_sites_clean:,} sites have a non-null pv_install_date.")
print(f"{install_dates.ac_capacity_kw.notna().sum():,} of those also have a non-null ac_capacity_kw.")

## 10. Table-level data volume

Row counts and on-disk size for the tables this project has actually built
so far -- fills in the bracketed placeholders still sitting in the Overleaf
draft's Section 3.


In [ ]:
def table_stats(name: str) -> dict:
    path = Config.store_path(name)
    files = list(path.rglob("*.parquet")) if path.is_dir() else ([path] if path.exists() else [])
    n_bytes = sum(f.stat().st_size for f in files)
    n_rows = con.sql(f"""
        SELECT count(*) FROM read_parquet('{path.as_posix()}{"/**/*.parquet" if path.is_dir() else ""}')
    """).fetchone()[0]
    return {"table": name, "n_rows": n_rows, "n_files": len(files), "size_gb": n_bytes / 1e9}

volume = pd.DataFrame([
    table_stats(name) for name in
    ["ami_raw", "ami_meter", "ami_raw_phaseseparate", "ami_meter_clean"]
])
volume.style.format({"n_rows": "{:,}", "size_gb": "{:.2f}"})


## 11. Per-site data-quality attrition

Not every site loses the same share of rows to notebook 01's hard checks --
this checks whether drops concentrate on a handful of sites (worth
excluding entirely) or spread evenly across the fleet (consistent with
generic sensor noise). Uses `ami_meter` (pre-cleaning) against
`ami_meter_clean`, a LEFT JOIN so a site dropped ENTIRELY (zero surviving
rows) shows up at 100% rather than silently disappearing.


In [ ]:
attrition = con.sql("""
    SELECT raw.site_id, raw.n_raw, COALESCE(clean.n_clean, 0) AS n_clean,
           1.0 - COALESCE(clean.n_clean, 0) * 1.0 / raw.n_raw AS drop_share
    FROM (SELECT site_id, count(*) AS n_raw FROM ami_meter GROUP BY site_id) raw
    LEFT JOIN (SELECT site_id, count(*) AS n_clean FROM ami_meter_clean GROUP BY site_id) clean
    USING (site_id)
""").df()

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(attrition.drop_share, bins=50, color=ACCENT)
ax.set_xlabel("Share of rows dropped (per site)")
ax.set_ylabel("Sites")
ax.set_title("Per-site attrition from ami_meter to ami_meter_clean")
sns.despine(ax=ax)
fig.tight_layout()

n_fully_dropped = int((attrition.n_clean == 0).sum())
print(f"{n_fully_dropped:,} site(s) lost ALL rows to hard-flagging (0% survive in ami_meter_clean).")
print("\nTop 10 sites by drop share:")
attrition.sort_values("drop_share", ascending=False).head(10)


## 12. Data-quality flags retained from notebook 01

`ami_meter_clean` keeps two soft-flagged, non-dropped columns for
transparency (see notebook 01) -- their prevalence here for reference, not
as a chart (two numbers don't need one).


In [ ]:
soft_flags = con.sql("""
    SELECT
      avg(CASE WHEN power_magnitude_extreme THEN 1.0 ELSE 0.0 END) AS power_magnitude_extreme_share,
      avg(CASE WHEN apparent_power_inconsistent THEN 1.0 ELSE 0.0 END) AS apparent_power_inconsistent_share
    FROM ami_meter_clean
""").df()
soft_flags.style.format("{:.4%}")


## 13. PV/load phase-count combinations

`n_load_phases`/`n_pv_phases` from `ami_site_metadata_clean` (derived in
notebook 06 from kept-circuit counts). A site's load/PV phase combination is
a real topology fact carried through from the resolution pipeline, not a
statistic computed fresh here.


In [ ]:
phase_combo = con.sql("""
    SELECT n_load_phases, n_pv_phases, count(*) AS n_sites
    FROM ami_site_metadata_clean
    GROUP BY n_load_phases, n_pv_phases
    ORDER BY n_sites DESC
""").df()

pivot_phase = phase_combo.pivot_table(
    index="n_load_phases", columns="n_pv_phases", values="n_sites", fill_value=0
).astype(int)
pivot_phase.index = [f"{n} load phase(s)" for n in pivot_phase.index]
pivot_phase.columns = [f"{n} PV phase(s)" for n in pivot_phase.columns]
pivot_phase.style.format("{:,}")


In [ ]:
combo_labels = [f"{int(r.n_load_phases)}L/{int(r.n_pv_phases)}P" for _, r in phase_combo.iterrows()]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(combo_labels, phase_combo.n_sites, color=ACCENT)
ax.set_xlabel("Load phases / PV phases")
ax.set_ylabel("Sites")
ax.set_title("PV/load phase-count combinations")
ax.tick_params(axis="x", rotation=45)
sns.despine(ax=ax)
fig.tight_layout()


## 14. Capacity: nameplate vs `S_99`

Two independent PV capacity proxies -- `ac_capacity_kw` (nameplate rating)
and `s_99` (empirical 99th-percentile apparent power, max-aggregated across
a site's circuits in notebook 06) -- used later for capacity-normalised
comparisons (per the paper's own Section 3). A scatter against the y=x line
shows how closely the two actually agree, rather than assuming they do.


In [ ]:
capacity = con.sql("""
    SELECT ac_capacity_kw, s_99
    FROM ami_site_metadata_clean
    WHERE ac_capacity_kw IS NOT NULL AND s_99 IS NOT NULL
""").df()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(capacity.ac_capacity_kw, capacity.s_99, s=8, alpha=0.3, color=ACCENT)
lims = [0, max(capacity.ac_capacity_kw.max(), capacity.s_99.max()) * 1.05]
ax.plot(lims, lims, color=OTHER_COLOR, linestyle="--", linewidth=1, label="y = x")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Nameplate ac_capacity_kw (kW)")
ax.set_ylabel("Empirical S_99 (kVA)")
ax.set_title("Nameplate vs empirical capacity")
ax.set_aspect("equal")
ax.legend(frameon=False)
sns.despine(ax=ax)
fig.tight_layout()

corr = capacity.ac_capacity_kw.corr(capacity.s_99)
print(f"{len(capacity):,} of {n_sites_clean:,} sites have both values. Pearson correlation: {corr:.3f}")


## 15. Manufacturer mix

Same `> DNSP_MIN_SITES`-site grouping rule as the DNSP section, applied to
`manufacturer` -- keeps the chart readable without hiding how concentrated
(or not) the fleet's inverter brands actually are. Full breakdown printed
alongside, same as the DNSP table.


In [ ]:
manufacturer_counts = con.sql("""
    SELECT manufacturer, count(*) AS n_sites
    FROM ami_site_metadata_clean
    WHERE manufacturer IS NOT NULL
    GROUP BY manufacturer ORDER BY n_sites DESC
""").df()

big_manufacturer = manufacturer_counts.loc[manufacturer_counts.n_sites > DNSP_MIN_SITES, "manufacturer"].tolist()
manufacturer_counts["manufacturer_group"] = manufacturer_counts.manufacturer.where(
    manufacturer_counts.manufacturer.isin(big_manufacturer), "Other"
)
manufacturer_order = big_manufacturer + (
    ["Other"] if (~manufacturer_counts.manufacturer.isin(big_manufacturer)).any() else []
)
grouped_manufacturer = manufacturer_counts.groupby("manufacturer_group", as_index=False).n_sites.sum()
grouped_manufacturer["manufacturer_group"] = pd.Categorical(
    grouped_manufacturer.manufacturer_group, categories=manufacturer_order, ordered=True
)
grouped_manufacturer = grouped_manufacturer.sort_values("manufacturer_group")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(grouped_manufacturer.manufacturer_group.astype(str), grouped_manufacturer.n_sites, color=ACCENT)
ax.set_ylabel("Sites")
ax.set_title(f"Inverter manufacturer mix (> {DNSP_MIN_SITES} sites shown individually)")
ax.tick_params(axis="x", rotation=45)
sns.despine(ax=ax)
fig.tight_layout()

print(f"Full breakdown, all {len(manufacturer_counts)} manufacturers (ami_meter_clean scope):")
manufacturer_counts[["manufacturer", "n_sites"]].style.format({"n_sites": "{:,}"})
